# 📝 LangChain 에이전트와 도구 과제 LV3(통합) — 리뷰 인텔리전스·조치 리포트

> 흩어진 리뷰(**비정형**)를 **구조화 수집 → pandas 집계 → 인사이트**로 잇고(1번), 거기서 나온 신호를 **데이터베이스의 사실**과 이어 조치 리포트까지 만듭니다(2번). 지금까지 따로 배운 스키마·부품 재사용·집계·Text-to-SQL 을 한 줄기로 꿰는 문제입니다.

## 풀이 방법
1. 맨 위 **준비 셀들**을 위에서부터 실행하세요. `.env` 에 본인 **`OPENAI_API_KEY`** 가 필요합니다.
2. 두 문제 모두 **여러 단계**로 나뉩니다. 단계마다 할 일이 셀에 적혀 있으니 순서대로 채우세요. **2번은 1번의 결과(`df`·`summary`)를 그대로 이어 씁니다** — 1번을 먼저 푸세요.
3. 모델이 만드는 답은 **실행할 때마다 달라져** 채점을 **타입·구조**로 합니다 — 어느 측면에 부정이 몇 건인지가 아니라, 집계 코드가 맞는지를 봅니다. 다만 **모델이 지어내면 안 되는 값**(집계 결과·DB 에서 조회한 재고)은 정확히 대조합니다.

화이팅!

아래 준비 셀을 먼저 실행하세요. 2번에서 쓸 데이터베이스는 그 문제 바로 앞에 따로 준비 셀이 있습니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 에이전트 공통 준비
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage

from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

## 1. 리뷰 인텔리전스 파이프라인
**배경**: 흩어진 도서 리뷰(**비정형**)를 측면별 감성으로 **구조화 수집**한 뒤, `pandas` 로 **집계**해 측면별 인사이트를 뽑는 파이프라인을 만듭니다. "비정형 → 정형 → 인사이트"의 실무 서사입니다.

제공 셀의 `ReviewBatch` 스키마와 지시문·리뷰 묶음을 씁니다.

In [ ]:
# [제공 코드] 측면별 감성 스키마 — 이 셀은 실행만 하세요.
from typing import Literal

from pydantic import BaseModel, Field


class AspectOpinion(BaseModel):
    aspect: Literal["배송", "품질", "가격", "내용"] = Field(description="리뷰가 언급한 측면")
    sentiment: Literal["긍정", "부정", "중립"] = Field(description="그 측면에 대한 감성")
    evidence: str = Field(description="그렇게 판단한 근거가 된 리뷰 속 표현")


class ReviewAspects(BaseModel):
    review_id: str = Field(description="리뷰 번호")
    opinions: list[AspectOpinion] = Field(description="이 리뷰가 다룬 측면별 의견 목록")


class ReviewBatch(BaseModel):
    results: list[ReviewAspects] = Field(description="여러 리뷰의 측면별 분석 결과 목록")


print("스키마 준비 완료")

In [ ]:
# [제공 코드] 리뷰 전체 로드 — 이 셀은 실행만 하세요.
import csv

_rows = list(csv.DictReader(open('data/bookstore_reviews.csv', encoding='utf-8')))
REVIEW_INSTR = (
    '다음 도서 리뷰들을 측면별로 분석하세요. 각 리뷰에서 언급된 측면(배송·품질·가격·내용)마다 감성(긍정·부정·중립)과 근거 표현을 뽑아 주세요. 언급되지 않은 측면은 포함하지 마세요.\n\n'
)
REVIEW_BLOCK = '\n'.join(f"[{r['review_id']}] {r['text']}" for r in _rows)
print('리뷰', len(_rows), '건 준비')

### 1단계 — 측면 감성 구조화 수집
`model.with_structured_output(ReviewBatch)` 로 **`REVIEW_INSTR + REVIEW_BLOCK`** 을 `invoke` 해 결과를 변수 **`batch`** 에 담으세요. 이어서 `(리뷰번호, 측면, 감성)` 튜플의 리스트 **`records`** 로 펼치세요.

리뷰 한 건이 여러 측면을 말할 수 있으니, `records` 의 길이는 리뷰 수보다 많아집니다. 순서는 **결과에 담긴 리뷰 순서, 그 안에서는 의견 순서** 그대로입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 모델에 스키마를 씌워 한 번 부르고, 결과를 두 겹 반복으로 납작하게 편다.

세부구현:
1. 지시문과 리뷰 묶음을 이어 붙인 하나의 문자열을 스키마 씌운 모델에 넘긴다.
2. 결과의 리뷰 목록을 돌고, 그 안에서 측면별 의견 목록을 다시 돈다.
3. 리뷰 번호·측면·감성 세 값을 튜플 하나로 만들어 차례로 모은다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(batch.results, list) and len(batch.results) >= 1
assert isinstance(records, list) and len(records) >= 1   # 측면 의견들이 펼쳐졌다
assert all(len(r) == 3 for r in records)                 # (리뷰번호, 측면, 감성) 세 칸
assert all(a in {'배송', '품질', '가격', '내용'} for _, a, _ in records)
assert all(s in {'긍정', '부정', '중립'} for _, _, s in records)
# 위 조건은 손으로 지어낸 튜플도 만족한다 — records 가 batch 를 펼친 것인지 그대로 대조한다
assert records == [(ra.review_id, o.aspect, o.sentiment)
                   for ra in batch.results for o in ra.opinions], \
    'batch 를 순서대로 펼친 결과여야 합니다'
print('✅ 통과!')

### 2단계 — pandas 집계
`records` 로 `DataFrame` 을 만들어 변수 **`df`** 에 담고(열: `review_id`, `aspect`, `sentiment`), **측면별 부정 건수**를 세어 변수 **`neg_by_aspect`**(딕셔너리: 측면→부정 건수)에 담으세요.

부정이 한 건도 없는 측면은 **열쇠 자체가 없어야** 합니다(0 으로 채워 넣지 않습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 튜플 목록으로 표를 만들고, 감성이 부정인 행만 남겨 측면별로 센다.

세부구현:
1. 튜플 리스트로 DataFrame 을 만들되 열 이름을 직접 지정한다.
2. 감성 열이 부정인 행만 걸러 낸다.
3. 그 행들의 측면 열을 값별로 세어 딕셔너리로 바꾼다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert {'review_id', 'aspect', 'sentiment'} <= set(df.columns), 'df 에 review_id·aspect·sentiment 세 열이 있어야 합니다'
assert len(df) == len(records)
assert isinstance(neg_by_aspect, dict)
assert set(neg_by_aspect) <= {'배송', '품질', '가격', '내용'}
assert all(int(v) >= 0 for v in neg_by_aspect.values())   # 건수(정수)
# 여기까지는 빈 딕셔너리도 통과한다 — df 의 부정 행과 실제로 대조한다
_neg_rows = df[df['sentiment'] == '부정']
assert set(neg_by_aspect) == set(_neg_rows['aspect']), '부정이 나온 측면이 빠졌거나 더 들어 있습니다'
assert sum(int(v) for v in neg_by_aspect.values()) == len(_neg_rows), '부정 건수의 합이 df 의 부정 행 수와 다릅니다'
print('✅ 통과!')

### 3단계 — 측면별 요약
측면별 **긍정·부정 건수**를 한눈에 보이게 정리합니다. `df` 를 측면×감성으로 **교차집계**해 변수 **`summary`** 에 담으세요(`pivot_table`, 값이 없으면 0). 행(index)은 **측면**, 열(columns)은 **감성**입니다. 어느 측면이 가장 **부정적**인지 눈으로 확인하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 앞 단계에서 만든 표를 측면(행)과 감성(열)으로 교차집계해 건수를 센다.

세부구현:
1. pivot_table 에 행 축과 열 축을 지정한다.
2. 세는 것이 목적이므로 집계 함수는 개수를 세는 것으로 하고, 값 열은 아무 열이나 준다.
3. 한 번도 나오지 않은 조합이 비어 있지 않도록 채울 값을 0 으로 지정한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert summary.index.name == 'aspect'
assert set(summary.index) <= {'배송', '품질', '가격', '내용'}
assert set(summary.columns) <= {'긍정', '부정', '중립'}
assert int(summary.to_numpy().sum()) == len(df)   # 교차집계는 전체 의견 수를 나눠 담는다
print('✅ 통과!')

## 2. 리뷰 인사이트를 데이터베이스 사실과 잇기
**배경**: 1번에서 **어느 측면에 불만이 몰렸는지**를 알았습니다. 그런데 그것만으로는 무엇을 할지 정할 수 없습니다. 불만이 **어느 책**에 몰렸는지는 리뷰에 있지만, 그 책이 지금 **얼마에 몇 권 남아 있는지**는 **데이터베이스**에 있습니다. 리뷰(비정형)에서 얻은 신호를 표(정형)의 사실과 이어 붙여 **조치 리포트**를 만듭니다.

1번의 `df` 를 그대로 이어 씁니다. 아래 제공 셀이 데이터베이스와 `run_select` 도구, 그리고 **리뷰 번호 → 도서 번호** 대응표(`REVIEW_BOOK`)를 준비합니다.

In [ ]:
# [제공 코드] 데이터베이스 준비 — SQL 단원에서 배운 sqlite 를 그대로 씁니다(접속 정보가 필요 없습니다).
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('output') / 'bookstore.db'
DB_PATH.parent.mkdir(exist_ok=True)
DB_PATH.unlink(missing_ok=True)          # 여러 번 실행해도 늘 같은 초기 상태에서 시작합니다

_conn = sqlite3.connect(DB_PATH, isolation_level=None)   # isolation_level=None : 실행 즉시 저장
_conn.execute('pragma foreign_keys = on')                # 외래키 검사를 켭니다(기본값은 꺼짐)
_conn.executescript(Path('data/setup_day19.sql').read_text(encoding='utf-8'))
_conn.execute('pragma foreign_keys = on')                # executescript 뒤에 한 번 더 켭니다

# 에이전트에게 줄 연결은 따로 만들고 '읽기 전용'으로 엽니다 — 모델이 무슨 SQL 을 만들든 쓰기가 막힙니다.
#  check_same_thread=False : 에이전트는 도구를 별도 스레드에서 실행하므로 이 옵션이 없으면 도구가 전부 실패합니다.
_ro_conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True,
                           isolation_level=None, check_same_thread=False)


def run_query(sql):
    """SELECT 결과를 DataFrame 으로 돌려준다(사람이 눈으로 확인할 때 쓴다)."""
    cur = _conn.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])


print('데이터베이스 준비 완료 —', DB_PATH)

In [ ]:
# [제공 코드] 데이터베이스 조회 도구 — 에이전트가 이 도구로 SQL 을 실행합니다.
#  가드가 두 겹입니다: (1) 여기서 문장을 검사하고 (2) 연결 자체가 읽기 전용입니다.
from langchain_core.tools import tool


@tool
def run_select(sql: str) -> str:
    """읽기 전용 SQL(SELECT) 한 문장을 실행하고 결과를 문자열로 돌려준다. SELECT 한 문장이 아니면 거부한다."""
    stmt = sql.strip().rstrip(';')          # 끝의 세미콜론 하나는 흔한 표기라 허용한다
    # 세미콜론이 남아 있으면 문장이 둘 이상이라는 뜻 — 'select 1; delete ...' 를 막는다.
    if not stmt.lower().startswith('select') or ';' in stmt:
        return '거부: 이 도구는 SELECT 조회 한 문장만 실행할 수 있습니다.'
    try:
        return str(_ro_conn.execute(stmt).fetchall())   # 검사한 문장을 그대로 실행한다
    except Exception as e:
        return f'에러: {e}'                             # 에러도 문자열로 — 모델이 읽고 고쳐 다시 시도한다


print('SQL 도구 준비:', run_select.name)

In [ ]:
# [제공 코드] 리뷰가 어느 책의 리뷰인지 — 이 셀은 실행만 하세요.
#  리뷰 파일에는 book_id 가 함께 들어 있습니다. 1번의 df 에는 리뷰 번호만 있으니 이 표로 이어 붙입니다.
REVIEW_BOOK = {r['review_id']: r['book_id'] for r in _rows}

# 에이전트에게 줄 표 설명 — 모델은 데이터베이스를 볼 수 없으므로 스키마를 글로 알려 줍니다.
BS_SCHEMA_PROMPT = (
    '너는 온라인 서점 데이터베이스를 조회해 답하는 도우미다. '
    '반드시 run_select 도구로 SELECT 를 실행해 확인한 값으로만 답한다. '
    'bs_book(book_id text, title text, author text, genre text, price int, stock int) 표가 있고, '
    'book_id 는 k1 처럼 생긴 도서 번호다. 결과를 한국어 한두 문장으로 정리해 답한다.'
)
print('대응표', len(REVIEW_BOOK), '건 / SQL 프롬프트 준비 완료')

### 1단계 — 불만이 가장 많이 몰린 책 찾기
`df` 에 **`book_id`** 열을 붙이고, 아래 셋을 구하세요.

| 변수 | 무엇 | 자료형 |
|---|---|---|
| **`worst_book`** | 부정 의견이 가장 많은 **도서 번호** | 문자열 |
| **`worst_count`** | 그 도서의 **부정 의견 건수** | **정수** |
| **`worst_aspect`** | **그 도서 안에서** 가장 많이 지적된 **측면** | 문자열 |

- `book_id` 는 `df['review_id']` 를 `REVIEW_BOOK` 으로 옮기면 됩니다.
- 부정 의견이 하나도 없는 책은 후보에서 빠집니다(세지 않은 책은 0 이 아니라 아예 없습니다).
- **동점이면 이름이 사전순으로 앞선 것**을 고르세요(도서 번호도, 측면도) — 규칙을 정해 두지 않으면 실행할 때마다 답이 달라집니다.
- ⚠️ **`worst_count` 는 파이썬 정수여야 합니다.** `value_counts()` 가 돌려주는 값은 파이썬 `int` 가 아니라 넘파이 정수(`numpy.int64`)라, 그대로 담으면 채점의 자료형 검사에 걸립니다 — **`int(...)` 로 감싸세요.** `worst_book`·`worst_aspect` 도 같은 이유로 `str(...)` 로 감쌉니다.

> 여기는 **모델이 개입하지 않는 순수 집계**입니다. 1번이 만든 `df` 만 정해지면 답은 하나로 정해집니다.

<details><summary>힌트</summary>

```text
접근방법:
- 리뷰 번호를 도서 번호로 옮겨 열 하나를 더한 뒤, 부정 행만 남겨 도서별로 센다.
- 그 책을 고른 다음, 그 책의 부정 행만 다시 추려 측면별로 센다.

세부구현:
1. df 의 review_id 열을 대응표로 옮겨 book_id 열을 만든다.
2. 감성이 부정인 행만 고른다.
3. 도서 번호별 개수를 세고, 개수가 가장 큰 것을 고르되 같으면 이름이 앞선 것을 고른다.
4. 고른 책의 부정 행만 다시 골라 측면별로 세고, 같은 규칙으로 하나를 고른다.
5. 세 값을 각각 str/int 로 감싸 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert 'book_id' in df.columns and df['book_id'].notna().all(), 'df 에 book_id 열을 붙이세요(빠진 리뷰가 없어야 합니다)'
assert isinstance(worst_book, str), 'worst_book 은 문자열이어야 합니다 — str(...) 로 감싸세요'
assert isinstance(worst_aspect, str), 'worst_aspect 는 문자열이어야 합니다 — str(...) 로 감싸세요'
# numpy.int64 는 파이썬 int 가 아니다 — value_counts() 결과를 int(...) 로 감쌌는지 본다
assert isinstance(worst_count, int), \
    'worst_count 는 파이썬 정수여야 합니다 — value_counts() 값은 numpy 정수라 int(...) 로 감싸세요'

# 손으로 적은 값이 아니라 df 에서 실제로 센 값인지 — 채점이 같은 규칙으로 다시 계산해 대조한다
_neg = df[df['sentiment'] == '부정']
# 건수는 큰 것부터, 같으면 이름이 앞선 것 -> (-건수, 이름) 이 가장 작은 것을 고르면 된다
_pick = lambda c: min(c.items(), key=lambda kv: (-kv[1], kv[0]))
_book, _count = _pick(_neg['book_id'].value_counts())
assert (worst_book, worst_count) == (_book, int(_count)), 'df 의 부정 행을 도서별로 센 결과와 다릅니다(동점이면 도서 번호가 앞선 것)'
_aspect, _ = _pick(_neg[_neg['book_id'] == worst_book]['aspect'].value_counts())
assert worst_aspect == _aspect, 'worst_aspect 는 그 도서의 부정 의견만 세어 고른 측면이어야 합니다(전체 통계가 아닙니다)'
print('✅ 통과!')

### 2단계 — 그 책의 사실을 데이터베이스에서 확인
`create_agent(model, [run_select], system_prompt=BS_SCHEMA_PROMPT)` 로 에이전트를 만들어 변수 **`sql_agent`** 에 담고, **`worst_book` 도서의 제목·가격·재고**를 묻는 질문으로 `invoke` 한 결과를 변수 **`res_sql`** 에 담으세요. 이어서 궤적에서 `ToolMessage` 만 골라 리스트 **`tool_msgs`** 에 담으세요.

- 질문 문자열에 **`worst_book` 값이 들어가야** 합니다(손으로 `'k2'` 라고 적지 말고 변수를 끼워 넣으세요). 1번의 결과가 바뀌면 질문도 따라 바뀌어야 하니까요.
- 모델이 어떤 SQL 을 만들지, 도구를 몇 번 부를지는 **실행마다 다릅니다.** 우리가 보장하는 것은 **도구가 실제로 불렸다**는 것과 **그 결과가 데이터베이스의 진짜 값**이라는 것뿐입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안에서 만든 Text-to-SQL 에이전트와 같은 모양이다. 질문에 도서 번호를 f-문자열로 끼워 넣는다.

세부구현:
1. 모델·도구 목록·시스템 프롬프트로 에이전트를 만든다.
2. worst_book 이 들어간 질문 문자열로 invoke 한다.
3. 결과의 messages 에서 ToolMessage 인 것만 골라 리스트로 만든다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert tool_msgs == [m for m in res_sql['messages'] if isinstance(m, ToolMessage)]
assert worst_book in res_sql['messages'][0].text, '질문에 worst_book 을 끼워 넣으세요'
assert len(tool_msgs) >= 1, 'run_select 가 한 번도 불리지 않았습니다'
assert all(m.name == 'run_select' for m in tool_msgs)
# 지문이 요구한 에이전트를 실제로 만들었는지 — 이름만 있고 안 쓰면 여기서 걸린다
assert hasattr(sql_agent, 'invoke'), 'sql_agent 에 create_agent 로 만든 에이전트를 담으세요'
# 궤적을 손으로 지어내지 않았는지 — 모델이 돌려준 AIMessage 에는 응답 메타데이터가 붙어 있다
assert any(isinstance(m, AIMessage) and getattr(m, 'response_metadata', None)
           for m in res_sql['messages']), \
    '모델이 실제로 돌려준 응답이 궤적에 없습니다 — sql_agent 를 정말 invoke 했는지 확인하세요'

# 도구가 돌려준 값이 진짜 DB 값인지 — 채점이 같은 것을 직접 조회해 대조한다(여기는 결정적이다)
_row = run_query(f"select title, price, stock from bs_book where book_id = '{worst_book}'").iloc[0]
_seen = ' '.join(str(m.content) for m in tool_msgs)
assert _row['title'] in _seen or str(_row['stock']) in _seen, '도구 결과에 그 책의 실제 값이 없습니다 — 다른 책을 조회했는지 확인하세요'
print('✅ 통과! 실제 값 —', dict(_row))

### 3단계 — 조치 리포트를 스키마로 받기
마지막으로 **사람에게 넘길 리포트**를 문장이 아니라 **데이터**로 받습니다. 아래 제공 셀의 `ActionReport` 스키마를 써서, 2단계와 **같은 도구·같은 프롬프트**에 **`response_format=ProviderStrategy(ActionReport, strict=True)`** 만 더한 에이전트를 만들어 변수 **`report_agent`** 에 담으세요. 그 에이전트를 아래 지시로 `invoke` 한 **결과 전체**를 변수 **`res_report`** 에 담고, 거기서 정형 결과를 꺼내 변수 **`report`** 에 담으세요(`report = res_report['structured_response']`).

지시 문장에는 **`worst_book`(도서 번호)**, **`worst_count`(부정 의견 건수)**, **`worst_aspect`(그 도서에서 가장 많이 지적된 측면)** 를 넣어, 모델이 리포트의 칸을 채울 수 있게 하세요. **`stock` 은 모델이 지어내면 안 되는 값**입니다 — 도구로 조회한 값이 들어가야 합니다.

> 결과를 **`res_report` 에 한 번 담아 두는 이유**가 있습니다. `report` 만 꺼내 버리면 **궤적(`res_report['messages']`)을 잃습니다** — 도구가 실제로 불렸는지 확인할 길이 사라지지요. 채점도 그 궤적을 봅니다.

> 임포트는 `from langchain.agents.structured_output import ProviderStrategy` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 2단계 에이전트에 response_format 인자 하나만 더한다. 도구 목록도 프롬프트도 그대로다.

세부구현:
1. 같은 인자에 response_format 을 더해 에이전트를 하나 더 만든다.
2. 도서 번호·부정 건수·측면이 들어간 지시 문장으로 invoke 하고, 그 결과를 통째로 변수에 담는다.
3. 그 결과 딕셔너리에서 structured_response 키를 꺼내 따로 담는다.
```

</details>

In [ ]:
# [제공 코드] 조치 리포트 스키마 — 2번 3단계에서 씁니다. 이 셀은 실행만 하세요.
from typing import Literal

from pydantic import BaseModel, Field


class ActionReport(BaseModel):
    """불만이 몰린 도서 한 권에 대한 조치 리포트."""

    book_id: str = Field(description="도서 번호(k1 처럼 생긴 값)")
    title: str = Field(description="도구로 조회한 도서 제목")
    top_aspect: Literal["배송", "품질", "가격", "내용"] = Field(
        description="그 도서에서 가장 많이 지적된 측면")
    negative_count: int = Field(description="그 도서의 부정 의견 건수")
    # 이 칸은 모델이 지어내는 값이 아니라 '도구가 알려 준 값'이다.
    stock: int = Field(description="도구로 조회한 현재 재고 수량")
    action: str = Field(description="담당자가 할 일 한 문장")


print("스키마 준비 완료:", list(ActionReport.model_fields))

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(report, ActionReport), 'structured_response 를 꺼냈는지 확인하세요'
assert report.book_id == worst_book, '리포트의 도서 번호가 1단계에서 고른 책과 다릅니다'
assert report.negative_count == worst_count
# 전체 통계가 아니라 '그 도서의' 최다 측면이어야 한다 — 1단계에서 구한 값과 같아야 한다
assert report.top_aspect == worst_aspect, 'top_aspect 는 그 도서에서 가장 많이 지적된 측면입니다(1단계의 worst_aspect)'
assert isinstance(report.action, str) and report.action.strip()

# stock 은 모델이 지어낼 값이 아니라 도구가 알려 준 값이다 — DB 와 정확히 같아야 한다
_stock = int(run_query(f"select stock from bs_book where book_id = '{worst_book}'").iloc[0, 0])
assert report.stock == _stock, f'재고가 DB 값({_stock})과 다릅니다 — 도구로 조회한 값을 넣어야 합니다'

# response_format 을 줘도 도구는 그대로 불린다 — 궤적이 남아 있는지 본다
assert any(isinstance(m, ToolMessage) for m in res_report['messages'])
print('✅ 통과!')

---
수고했어요! 흩어진 리뷰를 **구조화 수집 → 집계 → 인사이트**로 잇는 파이프라인을 완성했고, 거기서 나온 신호를 **데이터베이스의 사실과 이어** 조치 리포트까지 만들었습니다. 모델이 한 일은 **리뷰를 스키마에 맞춰 읽고, 필요한 SQL 을 만들어 도구에 넘기는 것**까지이고, 집계와 최종 판단의 근거가 된 값은 전부 **우리가 확인할 수 있는 사실**이었다는 점을 눈여겨보세요 — 실무 파이프라인이 대개 이 모양입니다. 다음 단원 **ReAct 에이전트** 에서는 에이전트가 도구를 고르는 **사고 루프의 원리**를 직접 해부합니다.